In [2]:
# Bibliotecas para trabalhar com os dados
import numpy as np
import pandas as pd
import pingouin as pg

# Seleção e avaliação dos modelos
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.model_selection import cross_validate, GridSearchCV

# Preparação dos dados
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

# Modelos preditivos
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor

# Métricas
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Gráficos
import plotly.express as px
import plotly.io as pio

#sistema
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 5)

pio.templates.default = 'plotly_dark'
px.defaults.height = 500
px.defaults.color_continuous_scale = px.colors.sequential.Viridis

### Leitura das transações
Preservamos os IDs como texto, convertemos `t_dat` para data e arredondamos os preços para quatro casas. `N_LINHAS` seleciona as primeiras linhas do arquivo, não uma amostra aleatória. Execute o notebook a partir desta pasta, pois as leituras de clientes e artigos abaixo usam caminhos relativos.

In [3]:
PASTA_DADOS = Path(".")
if not (PASTA_DADOS / "transactions_train.csv").is_file():
    PASTA_DADOS = Path("H&M - Hackaton")

N_LINHAS = 1_000_000  # None para ler o arquivo completo.

df_transaction = pd.read_csv(
    PASTA_DADOS / "transactions_train.csv",
    index_col=False,
    dtype={"customer_id": "string", "article_id": "string"},
    nrows=N_LINHAS,
)

df_transaction['price'] = df_transaction['price'].astype(float).round(4)

df_transaction['t_dat'] = pd.to_datetime(df_transaction['t_dat'], format='%Y-%m-%d')

df_transaction

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.0508,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.0305,2
...,...,...,...,...,...
999998,2018-10-11,46193b292b8c4cbef8d2b75a75b92584c4fef588b07171...,0524825019,0.0339,2
999999,2018-10-11,461d77eece32034c78a9c1425f84b707bffed6d86368c0...,0461414008,0.0393,2


### Cadastro de clientes e primeiro merge
O `left` mantém as transações e `validate='many_to_one'` impede duplicações causadas por IDs repetidos no cadastro. Atenção: limitar clientes e transações ao mesmo número de linhas não seleciona as mesmas pessoas. Mais abaixo recuperamos o cadastro completo em uma cópia para exploração.

In [4]:

df_clientes = pd.read_csv(
    "customers.csv",
    index_col=False,
    dtype={"customer_id": "string"},
    nrows = N_LINHAS
)

display(df_clientes)

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
...,...,...,...,...,...,...,...
999998,ba9ade47e568d76b62f274b023fbc73acfba04e38c1082...,1.0,1.0,ACTIVE,Regularly,62.0,0c9f3dc39438e945c5c8fbafbbba0d67b3fd627f667084...
999999,ba9ae8434966611eb42e03e0bfca7eac3e7e1c16dbc210...,NaN,NaN,ACTIVE,NONE,32.0,caaebcb2559fb0b6148b762b2341fc9064936aa1baf4bc...


In [5]:
df_main = df_transaction.merge(
      df_clientes[['customer_id', 'club_member_status']]
    , on='customer_id'
    , how='left'
    , validate='many_to_one'
)

df_main

,t_dat,customer_id,article_id,price,sales_channel_id,club_member_status
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.0508,2,ACTIVE
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.0305,2,ACTIVE
...,...,...,...,...,...,...
999998,2018-10-11,46193b292b8c4cbef8d2b75a75b92584c4fef588b07171...,0524825019,0.0339,2,ACTIVE
999999,2018-10-11,461d77eece32034c78a9c1425f84b707bffed6d86368c0...,0461414008,0.0393,2,ACTIVE


### Cadastro de artigos e segundo merge
Os textos foram padronizados em minúsculas, sem espaços nas pontas e com `_` entre palavras. O `.dropna()` dentro da atribuição não exclui linhas de `df_articles`: o pandas realinha pelo índice e mantém os valores ausentes. A descrição foi removida e o grupo do produto foi acrescentado ao `df_main`.

In [6]:
df_articles = pd.read_csv(
    "articles.csv",
    index_col=False,
    dtype={"article_id": "string"},
    nrows=N_LINHAS
)

df_articles.drop(columns=['detail_desc'], inplace=True)

for column in ['prod_name', 'product_type_name', 'product_group_name', 'graphical_appearance_name', 'colour_group_name', 'perceived_colour_value_name', 'perceived_colour_master_name', 'department_name', 'index_name', 'index_group_name', 'section_name', 'garment_group_name']:
    df_articles[column] = (
            df_articles[column]
            .astype("string")
            .dropna()
            .str.lower()
            .str.strip()
            .str.replace(r"\s+", "_", regex=True)
    )

df_articles

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name
0,0108775015,108775,strap_top,253,vest_top,garment_upper_body,1010016,solid,9,black,4,dark,5,black,1676,jersey_basic,A,ladieswear,1,ladieswear,16,womens_everyday_basics,1002,jersey_basic
1,0108775044,108775,strap_top,253,vest_top,garment_upper_body,1010016,solid,10,white,3,light,9,white,1676,jersey_basic,A,ladieswear,1,ladieswear,16,womens_everyday_basics,1002,jersey_basic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105540,0957375001,957375,claire_hair_claw,72,hair_clip,accessories,1010016,solid,9,black,4,dark,5,black,3946,small_accessories,D,divided,2,divided,52,divided_accessories,1019,accessories
105541,0959461001,959461,lounge_dress,265,dress,garment_full_body,1010016,solid,11,off_white,1,dusty_light,9,white,1641,jersey,A,ladieswear,1,ladieswear,18,womens_trend,1005,jersey_fancy


In [7]:
df_main = df_main.merge(
      df_articles[['article_id', 'product_group_name']]
    , on='article_id'
    , how='left'
    , validate='many_to_one'
)

df_main

,t_dat,customer_id,article_id,price,sales_channel_id,club_member_status,product_group_name
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.0508,2,ACTIVE,underwear
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.0305,2,ACTIVE,underwear
...,...,...,...,...,...,...,...
999998,2018-10-11,46193b292b8c4cbef8d2b75a75b92584c4fef588b07171...,0524825019,0.0339,2,ACTIVE,garment_lower_body
999999,2018-10-11,461d77eece32034c78a9c1425f84b707bffed6d86368c0...,0461414008,0.0393,2,ACTIVE,shoes


### Conferências e gráficos iniciais
`describe()` e `info()` mostram distribuição, tipos e ausências. Nos gráficos, a contagem representa **linhas de itens vendidos**, não pedidos distintos: não temos um identificador de pedido. O último dia pode estar incompleto por causa de `N_LINHAS`.

In [8]:
with pd.option_context("display.max_rows", None):
    display(df_main.describe())
    df_main.info()


,t_dat,price,sales_channel_id
count,1000000,1000000.000000,1000000.000000
mean,2018-09-29 21:07:43.190400256,0.030177,1.662943
min,2018-09-20 00:00:00,0.000700,1.000000
25%,2018-09-25 00:00:00,0.016900,1.000000
50%,2018-09-29 00:00:00,0.025400,2.000000
75%,2018-10-05 00:00:00,0.033900,2.000000
max,2018-10-11 00:00:00,0.591500,2.000000
std,NaN,0.022915,0.472705


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 7 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   t_dat               1000000 non-null  datetime64[ns]
 1   customer_id         1000000 non-null  string        
 2   article_id          1000000 non-null  string        
 3   price               1000000 non-null  float64       
 4   sales_channel_id    1000000 non-null  int64         
 5   club_member_status  729572 non-null   object        
 6   product_group_name  1000000 non-null  string        
dtypes: datetime64[ns](1), float64(1), int64(1), object(1), string(3)
memory usage: 53.4+ MB


In [9]:
import plotly.express as px

resumo = (
    df_main["product_group_name"]
    .value_counts()
    .reset_index()
)

fig = px.bar(
    resumo,
    x="product_group_name",
    y="count",
    title="Quantidade de artigos vendidos por categoria",
    labels={
        "product_group_name": "Categoria",
        "count": "Quantidade vendida"
    }
)

fig.show()


compras_dia = (
    df_main.groupby("t_dat")
    .size()
    .reset_index(name="qtd_compras")
)

fig = px.line(
    compras_dia,
    x="t_dat",
    y="qtd_compras",
    title="Evolução diária das compras"
)

fig.show()

### Estatísticas de preço
Correção pontual: `.agg()` recebe as funções em uma **lista**. Mantivemos as mesmas cinco estatísticas; antes, `'mean'` era interpretado como argumento de eixo e gerava erro.

In [9]:
estatisticas_preco = df_main['price'].agg([
      'count'
    , 'mean'
    , 'median'
    , 'std'
    , 'skew'
])

display(estatisticas_preco)

count     1000000.000000
mean            0.030177
median          0.025400
std             0.022915
skew            3.645124
Name: price, dtype: float64

### Correlações exploratórias — preparação
Vamos trabalhar em `df_exploracao`, mantendo o `df_main` como foi montado. Relemos apenas ID e status de **todos** os clientes para corrigir as ausências geradas pelo limite do cadastro. Status realmente ausentes continuam ausentes, sem inventar uma categoria.

As correlações abaixo usam preço e canal; status e grupo de produto ficam disponíveis para futuras comparações por categoria.

In [ ]:
df_cadastro = pd.read_csv(
    PASTA_DADOS / 'customers.csv',
    usecols=['customer_id', 'club_member_status'],
    dtype={'customer_id': 'string'}
)

df_exploracao = df_main.drop(columns=['club_member_status']).merge(
      df_cadastro
    , on='customer_id'
    , how='left'
    , validate='many_to_one'
    , indicator='cadastro'
)

print('Linhas antes e depois:',                 len(df_main), len(df_exploracao))
print('Status ausentes no df_main:',            df_main['club_member_status'].isna().sum())
print('Status ausentes após reler o cadastro:', df_exploracao['club_member_status'].isna().sum())
print('Linhas sem cliente no cadastro:',        df_exploracao['cadastro'].eq('left_only').sum())
print('Linhas sem grupo de produto:',           df_exploracao['product_group_name'].isna().sum())

Linhas antes e depois: 1000000 1000000
Status ausentes no df_main: 270428
Status ausentes após reler o cadastro: 10235
Linhas sem cliente no cadastro: 0
Linhas sem grupo de produto: 0


### O que entra na correlação
**Pearson** mede relação linear; **Spearman** mede se as variáveis tendem a crescer ou diminuir juntas pela ordem dos valores. Ambos variam de -1 a 1. Valores próximos de zero não descartam outros tipos de relação.

IDs, datas e nomes de categorias não entram como números. O canal vira `canal_2`: **1 = canal 2; 0 = canal 1**. Com dois canais, Pearson nessa variável binária mede a associação entre preço e grupo (correlação ponto-bisserial); não significa que um canal seja “maior” que o outro. Não atribuímos nomes aos canais sem confirmar o dicionário dos dados.

Usamos as mesmas linhas válidas nas duas análises: cliente identificado, preço finito e canal 1 ou 2. Informamos qualquer exclusão. Mantemos os preços arredondados que você preparou; isso pode criar empates, inclusive no Spearman. `price` permanece na escala do arquivo, sem atribuir moeda.

In [11]:
if not df_exploracao['sales_channel_id'].dropna().isin([1, 2]).all():
    raise ValueError('Há outros canais na base. Revise a comparação entre os canais 1 e 2.')

df_analise = df_exploracao.loc[
      df_exploracao['customer_id'].notna()
    & np.isfinite(df_exploracao['price'])
    & df_exploracao['sales_channel_id'].isin([1, 2]),
    ['customer_id', 'price', 'sales_channel_id']
].copy()

df_analise['canal_2'] = df_analise['sales_channel_id'].eq(2).astype(int)

periodo = 'Período sem datas válidas'
if df_exploracao['t_dat'].notna().any():
    periodo = f"{df_exploracao['t_dat'].min():%d/%m/%Y} a {df_exploracao['t_dat'].max():%d/%m/%Y}"
print('Período do recorte:', periodo)
print('Linhas usadas:', len(df_analise))
print('Linhas excluídas:', len(df_exploracao) - len(df_analise))

resumo_canal = df_analise.groupby('sales_channel_id').agg(
      qtd_itens=('price', 'size')
    , qtd_clientes=('customer_id', 'nunique')
    , preco_medio=('price', 'mean')
    , preco_mediano=('price', 'median')
    , desvio_preco=('price', 'std')
)

display(resumo_canal.round(4))

Período do recorte: 20/09/2018 a 11/10/2018
Linhas usadas: 1000000
Linhas excluídas: 0


,qtd_itens,qtd_clientes,preco_medio,preco_mediano,desvio_preco
sales_channel_id,,,,,
1,337057,107336,0.0265,0.0244,0.0178
2,662943,124927,0.0321,0.0254,0.0249


### Pearson e Spearman por item vendido
Cada linha tem o mesmo peso; clientes com mais itens aparecem mais vezes. Sinal positivo entre `price` e `canal_2` indica associação com preços maiores no canal 2; sinal negativo, com preços menores. Isso pode refletir diferenças nos produtos vendidos, não um efeito do canal.

A escala de cores é fixa de -1 a 1 nas quatro matrizes. Células vazias/`NaN` indicam correlação indefinida (por exemplo, coluna constante ou menos de três observações), não correlação zero. O mínimo de três apenas evita pares insuficientes; não garante evidência estatística.

In [12]:
corr_pearson = df_analise[['price', 'canal_2']].corr(method='pearson', min_periods=3)
corr_spearman = df_analise[['price', 'canal_2']].corr(method='spearman', min_periods=3)

for metodo, matriz in [('Pearson', corr_pearson), ('Spearman', corr_spearman)]:
    fig = px.imshow(
        matriz,
        text_auto='.2f',
        zmin=-1,
        zmax=1,
        color_continuous_scale=['#3274A1', '#F3F3F3', '#C97932'],
        labels={'color': 'Correlação'},
        title=f'{metodo} — preço e canal por item<br><sup>{periodo} | {len(df_analise):,} itens</sup>'
    )
    fig.update_layout(height=500, margin=dict(t=90, b=70))
    fig.show()

### Perfil observado por cliente
Agora cada cliente ocupa uma linha e tem o mesmo peso nas matrizes. Calculamos quantidade de itens, preço médio, mediana, desvio dos preços e proporção de itens no canal 2.

`proporcao_canal_2 = 0` significa só canal 1 no recorte; `1`, só canal 2; `0.5`, metade dos itens em cada canal. A mediana resume o preço habitual **observado** e o desvio mostra sua dispersão. Não são medidas de fidelidade futura nem de preço por pedido.

Para comparar a dispersão, usamos clientes com pelo menos **dois itens**. Isso não exige duas visitas: os itens podem ser do mesmo dia. Clientes com apenas um item ficam em `perfil_clientes`, mas fora dessas matrizes. Média e mediana descrevem o mesmo preço, então uma correlação alta entre elas é esperada.

In [13]:
perfil_clientes = df_analise.groupby('customer_id').agg(
      qtd_itens=('price', 'size')
    , preco_medio=('price', 'mean')
    , preco_mediano=('price', 'median')
    , desvio_preco=('price', 'std')
    , proporcao_canal_2=('canal_2', 'mean')
).reset_index()

MIN_ITENS = 2
perfil_correlacao = perfil_clientes.loc[perfil_clientes['qtd_itens'] >= MIN_ITENS].copy()

print('Clientes no recorte válido:', len(perfil_clientes))
print('Clientes nas matrizes:', len(perfil_correlacao))
print('Clientes abaixo do mínimo de itens:', len(perfil_clientes) - len(perfil_correlacao))

display(perfil_clientes.head())

Clientes no recorte válido: 214936
Clientes nas matrizes: 167591
Clientes abaixo do mínimo de itens: 47345


,customer_id,qtd_itens,preco_medio,preco_mediano,desvio_preco,proporcao_canal_2
0,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,8,0.036425,0.02965,0.020234,0.875
1,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,3,0.044033,0.05080,0.011720,1.000
2,00007d2de826758b65a93dd24ce629ed66842531df6699...,23,0.015961,0.01520,0.007881,1.000
3,00009c2aeae8761f738e4f937d9be6b49861a66339c2b1...,1,0.025400,0.02540,NaN,0.000
4,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,6,0.029367,0.02710,0.003512,1.000


In [14]:
colunas_correlacao = [
      'qtd_itens'
    , 'preco_medio'
    , 'preco_mediano'
    , 'desvio_preco'
    , 'proporcao_canal_2'
]

corr_clientes_pearson = perfil_correlacao[colunas_correlacao].corr(method='pearson', min_periods=3)
corr_clientes_spearman = perfil_correlacao[colunas_correlacao].corr(method='spearman', min_periods=3)

rotulos = ['Qtd. itens', 'Preço médio', 'Preço mediano', 'Desvio do preço', 'Proporção canal 2']

for metodo, matriz in [('Pearson', corr_clientes_pearson), ('Spearman', corr_clientes_spearman)]:
    fig = px.imshow(
        matriz,
        x=rotulos,
        y=rotulos,
        text_auto='.2f',
        zmin=-1,
        zmax=1,
        color_continuous_scale=['#3274A1', '#F3F3F3', '#C97932'],
        labels={'color': 'Correlação'},
        title=f'{metodo} — perfil dos clientes<br><sup>{periodo} | {len(perfil_correlacao):,} clientes com pelo menos {MIN_ITENS} itens</sup>'
    )
    fig.update_layout(height=600, margin=dict(t=90, b=110))
    fig.update_xaxes(tickangle=-25)
    fig.show()

### Hipóteses para investigar depois
- Quem concentra itens no canal 2 apresenta preço mediano diferente? Compare `preco_mediano` com `proporcao_canal_2` nos dois métodos.
- Clientes com mais itens apresentam maior ou menor dispersão de preços? Compare `qtd_itens` com `desvio_preco`; o número de observações também influencia a estimativa da dispersão.
- A relação entre preço e canal continua ao comparar o mesmo grupo de produto? Essa comparação ajuda a separar associação com canal de diferenças no mix de produtos.

Essas matrizes levantam hipóteses; não testam causalidade, significância ou permanência de preferência. O recorte inicial é curto, termina possivelmente no meio do dia e contém itens repetidos por cliente. Correlação por item e por cliente responde a perguntas diferentes. Antes de modelar, será necessário definir o alvo e separar passado e futuro: um resumo calculado com todo o período pode vazar informação futura.

Referências de código: [pandas — correlações](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html) e [Plotly — heatmaps anotados](https://plotly.com/python/annotated-heatmap/).

### Preferência de cada cliente por categoria
A pergunta agora é: **quais categorias ocupam mais espaço no histórico daquele cliente, e isso se repete em outros dias?** `product_group_name` representa um grupo amplo, não um artigo específico.

Usamos apenas cliente, categoria e data do `df_main`, sem filtrar preço, canal ou status. Mantemos as linhas de itens vendidos: repetições podem representar várias unidades, por isso não removemos duplicatas sem um ID de pedido. Categorias ausentes, vazias ou `unknown` ficam fora, com a quantidade informada.

Como `N_LINHAS` pode cortar um dia no meio, retiramos **somente desta EDA** o último dia quando o limite de leitura foi atingido. É uma precaução: pode remover um dia completo, mas evita comparar um dia possivelmente incompleto. Os DataFrames anteriores permanecem iguais.

In [15]:
df_preferencia = df_main[['customer_id', 'product_group_name', 't_dat']].copy()
df_preferencia['dia'] = df_preferencia['t_dat'].dt.normalize()

linhas_validas = (
      df_preferencia[['customer_id', 'product_group_name', 'dia']].notna().all(axis=1)
    & df_preferencia['product_group_name'].str.strip().ne('')
    & df_preferencia['product_group_name'].ne('unknown')
)
print('Linhas sem cliente, data ou categoria conhecida:', (~linhas_validas).sum())
df_preferencia = df_preferencia.loc[linhas_validas].copy()

if N_LINHAS is not None and len(df_transaction) >= N_LINHAS:
    ultimo_dia = df_main['t_dat'].max().normalize()
    print('Itens retirados do último dia:', df_preferencia['dia'].eq(ultimo_dia).sum())
    df_preferencia = df_preferencia.loc[df_preferencia['dia'] < ultimo_dia].copy()

if df_preferencia.empty:
    raise ValueError('Não há histórico válido para explorar preferências neste recorte.')

periodo_preferencia = f"{df_preferencia['dia'].min():%d/%m/%Y} a {df_preferencia['dia'].max():%d/%m/%Y}"
print('Período desta EDA:', periodo_preferencia)
print('Itens:', len(df_preferencia))
print('Clientes:', df_preferencia['customer_id'].nunique())
print('Categorias:', df_preferencia['product_group_name'].nunique())

Linhas sem cliente, data ou categoria conhecida: 607
Itens retirados do último dia: 16404
Período desta EDA: 20/09/2018 a 10/10/2018
Itens: 982989


Clientes: 212497
Categorias: 14


### Participação e afinidade por categoria
`participacao_cliente` é a fração dos itens do cliente naquela categoria; suas participações somam 100%. `participacao_dias` é a fração dos dias ativos em que ele comprou a categoria. Como um dia pode ter várias categorias, as participações por dia podem somar mais de 100%.

`afinidade_relativa` divide a participação do cliente pela participação da categoria em **todos os itens** deste recorte. Exemplo: 40% no cliente e 20% na base resulta em 2 — concentração duas vezes maior que a geral. Isso não é probabilidade, causalidade ou teste estatístico; categorias raras podem produzir valores grandes com poucos itens. Por isso mostramos também quantidade e dias.

A categoria líder será a de maior participação, não a de maior afinidade. Empates serão preservados. Zero na matriz significa ausência de compra observada, não rejeição.

In [16]:
perfil_preferencia = df_preferencia.groupby('customer_id').agg(
      total_itens=('product_group_name', 'size')
    , dias_ativos=('dia', 'nunique')
    , qtd_categorias=('product_group_name', 'nunique')
)

popularidade_categorias = (
    df_preferencia['product_group_name']
    .value_counts(normalize=True)
    .rename('participacao_geral')
)

preferencias_clientes = df_preferencia.groupby(['customer_id', 'product_group_name']).agg(
      qtd_itens=('product_group_name', 'size')
    , dias_categoria=('dia', 'nunique')
).reset_index()

preferencias_clientes['participacao_cliente'] = (
    preferencias_clientes['qtd_itens']
    / preferencias_clientes['customer_id'].map(perfil_preferencia['total_itens'])
)
preferencias_clientes['participacao_dias'] = (
    preferencias_clientes['dias_categoria']
    / preferencias_clientes['customer_id'].map(perfil_preferencia['dias_ativos'])
)
preferencias_clientes['participacao_geral'] = preferencias_clientes['product_group_name'].map(popularidade_categorias)
preferencias_clientes['afinidade_relativa'] = preferencias_clientes['participacao_cliente'] / preferencias_clientes['participacao_geral']
preferencias_clientes['posicao'] = preferencias_clientes.groupby('customer_id')['qtd_itens'].rank(method='min', ascending=False)

categorias_lideres = preferencias_clientes.loc[preferencias_clientes['posicao'] == 1].copy()
perfil_preferencia['participacao_lider'] = categorias_lideres.groupby('customer_id')['participacao_cliente'].max()
perfil_preferencia['qtd_lideres'] = categorias_lideres.groupby('customer_id').size()

# Se houver empate, a categoria líder única fica ausente.
lideres_unicas = categorias_lideres.drop_duplicates('customer_id', keep=False)
perfil_preferencia['categoria_lider'] = lideres_unicas.set_index('customer_id')['product_group_name']

matriz_preferencia = preferencias_clientes.pivot(
      index='customer_id'
    , columns='product_group_name'
    , values='participacao_cliente'
).fillna(0)

display(preferencias_clientes.head().round(3))

,customer_id,product_group_name,qtd_itens,dias_categoria,participacao_cliente,participacao_dias,participacao_geral,afinidade_relativa,posicao
0,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,garment_full_body,2,2,0.250,0.500,0.061,4.076,2.0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,garment_lower_body,1,1,0.125,0.250,0.232,0.54,3.0
2,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,garment_upper_body,5,4,0.625,1.000,0.5,1.251,1.0
3,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,underwear,3,2,1.000,1.000,0.064,15.632,1.0
4,00007d2de826758b65a93dd24ce629ed66842531df6699...,accessories,3,1,0.130,0.333,0.053,2.456,4.0


### Quanto histórico sustenta essa preferência?
Um cliente com um único item sempre terá concentração de 100%. Para não confundir pouco histórico com preferência bem definida, usamos inicialmente **pelo menos cinco itens em dois dias diferentes**. É um filtro exploratório ajustável, não uma garantia estatística. Todos continuam nas tabelas completas; o filtro só define quem entra nos gráficos de comparação.

A tabela abaixo mostra como a concentração muda ao exigir mais histórico. Cada cliente tem o mesmo peso.

In [17]:
MIN_ITENS_PREFERENCIA = 5
MIN_DIAS_PREFERENCIA = 2

clientes_com_historico = perfil_preferencia.loc[
      (perfil_preferencia['total_itens'] >= MIN_ITENS_PREFERENCIA)
    & (perfil_preferencia['dias_ativos'] >= MIN_DIAS_PREFERENCIA)
].copy()

resumo_suporte = []
for min_itens, min_dias in [(1, 1), (5, 1), (5, 2), (10, 2)]:
    grupo = perfil_preferencia.loc[
          (perfil_preferencia['total_itens'] >= min_itens)
        & (perfil_preferencia['dias_ativos'] >= min_dias)
    ]
    resumo_suporte.append({
          'min_itens': min_itens
        , 'min_dias': min_dias
        , 'clientes': len(grupo)
        , 'cobertura_pct': 100 * len(grupo) / len(perfil_preferencia)
        , 'concentracao_mediana_pct': 100 * grupo['participacao_lider'].median()
        , 'lider_unica_pct': 100 * grupo['qtd_lideres'].eq(1).mean()
    })

resumo_suporte = pd.DataFrame(resumo_suporte)
display(resumo_suporte.round(1))

if not clientes_com_historico.empty:
    fig = px.histogram(
        clientes_com_historico,
        x='participacao_lider',
        title=f'Concentração na categoria líder<br><sup>{periodo_preferencia} | {len(clientes_com_historico):,} clientes; mínimo {MIN_ITENS_PREFERENCIA} itens / {MIN_DIAS_PREFERENCIA} dias</sup>',
        labels={'participacao_lider': 'Participação da categoria líder'},
        color_discrete_sequence=['#3274A1']
    )
    fig.update_traces(xbins=dict(start=0, end=1.0001, size=0.05))
    fig.update_xaxes(tickformat='.0%', range=[0, 1.02])
    fig.update_yaxes(title='Clientes')
    fig.update_layout(margin=dict(t=100))
    fig.show()
else:
    print('Nenhum cliente atingiu o histórico mínimo escolhido.')

,min_itens,min_dias,clientes,cobertura_pct,concentracao_mediana_pct,lider_unica_pct
0,1,1,212497,100.0,80.0,84.5
1,5,1,71961,33.9,60.0,88.6
2,5,2,31492,14.8,60.0,89.0
3,10,2,12675,6.0,56.7,92.8


### Comparando perfis individuais
Selecionamos até dez clientes aleatórios com o histórico mínimo, usando uma semente fixa. Não são os maiores compradores nem uma amostra para inferência; servem para visualizar diferenças individuais. Na figura, **cada coluna é um cliente e soma 100%**. A tabela relaciona os rótulos aos IDs reais e mostra o suporte de cada perfil.

In [18]:
amostra_preferencias = clientes_com_historico.sample(
    n=min(10, len(clientes_com_historico)), random_state=42
).copy()
amostra_preferencias['rotulo'] = [f'Cliente {i + 1}' for i in range(len(amostra_preferencias))]

if not amostra_preferencias.empty:
    matriz_exemplo = matriz_preferencia.loc[amostra_preferencias.index, popularidade_categorias.index].T
    fig = px.imshow(
        matriz_exemplo,
        x=amostra_preferencias['rotulo'].tolist(),
        y=matriz_exemplo.index.str.replace('_', ' ').tolist(),
        zmin=0,
        zmax=1,
        aspect='auto',
        color_continuous_scale=['#F3F3F3', '#3274A1'],
        labels={'color': 'Participação'},
        title=f'Categorias no histórico de cada cliente<br><sup>{periodo_preferencia} | amostra ilustrativa com histórico mínimo; cada coluna soma 100%</sup>'
    )
    # Mostrar os percentuais observados e deixar os zeros sem texto.
    fig.update_traces(
        text=np.where(matriz_exemplo > 0, (matriz_exemplo * 100).round().astype(int).astype(str) + '%', ''),
        texttemplate='%{text}',
        hovertemplate='%{x}<br>%{y}<br>Participação: %{z:.1%}<extra></extra>'
    )
    fig.update_layout(height=700, margin=dict(t=100, l=160, b=70), coloraxis_colorbar_tickformat='.0%')
    fig.update_xaxes(title='Cliente')
    fig.update_yaxes(title='Categoria')
    fig.show()
    with pd.option_context('display.max_rows', 10):
        display(amostra_preferencias[['rotulo', 'total_itens', 'dias_ativos', 'categoria_lider', 'qtd_lideres']])

,rotulo,total_itens,dias_ativos,categoria_lider,qtd_lideres
customer_id,,,,,
7f8c345bdeff3d1f17aaccf9b5fe01a2c20cda41e805b4eeaa1ef695cc3d9d0d,Cliente 1,11,2,garment_lower_body,1
d72d1e39602e620b5c19a98f9f2965a3c3a4f18a53bef66cdaadd9fccf36254d,Cliente 2,6,4,garment_upper_body,1
38f20a0ecdf2afbed6e2747062c66c1c52138da6857b5301aa0253bb491f619a,Cliente 3,5,2,garment_upper_body,1
30949664ff949c8a93b13abf10925fdd3a22cc9b423942baa15f3b1c5218cbc9,Cliente 4,80,8,garment_lower_body,1
09015e1b02f1878c1641991a4e09fcb5ba1df74b5012f463d60a4186697691a7,Cliente 5,6,2,<NA>,2
91bb6a989e34b5a36e21f36f7b0b79fb124b189b1318fcb3cae6e54adbf40d25,Cliente 6,5,2,garment_upper_body,1
336b24ed07ea056f8f6fa784c716cf55b21341d480532d2266ca275b307f0227,Cliente 7,6,4,garment_upper_body,1
58fa42b025f4708234040c26d9de1081b603d1acda5c20c6095e3f291e5d560a,Cliente 8,15,2,garment_upper_body,1
e51a3f0c994e8b7e0b0f5ab83fdbf92639fe77067531e11c3337f2091ac1a4f0,Cliente 9,27,3,garment_upper_body,1


### Olhando aquele cliente
Troque `CLIENTE_ID` pelo ID que quiser investigar. Por padrão, usamos o primeiro cliente da amostra acima. A tabela detalha suas categorias; o gráfico compara a participação no cliente com a participação geral. A referência geral é ponderada por itens, enquanto o perfil é normalizado dentro do cliente.

Uma categoria com afinidade alta, mas presente em apenas um dia, ainda pode refletir uma ocasião específica. A leitura mais útil combina participação, afinidade e repetição em dias diferentes. Um empate não é resolvido escolhendo arbitrariamente uma categoria.

In [19]:
CLIENTE_ID = amostra_preferencias.index[0] if not amostra_preferencias.empty else None

if CLIENTE_ID in perfil_preferencia.index:
    preferencia_cliente = preferencias_clientes.loc[
        preferencias_clientes['customer_id'] == CLIENTE_ID
    ].sort_values(['participacao_cliente', 'product_group_name'], ascending=[False, True])

    print('Cliente:', CLIENTE_ID)
    print('Itens:', perfil_preferencia.loc[CLIENTE_ID, 'total_itens'])
    print('Dias ativos:', perfil_preferencia.loc[CLIENTE_ID, 'dias_ativos'])
    print('Categorias na liderança:', perfil_preferencia.loc[CLIENTE_ID, 'qtd_lideres'])
    with pd.option_context('display.max_rows', None):
        display(preferencia_cliente.drop(columns=['customer_id']).round(3))

    comparacao_cliente = pd.DataFrame({
          'Este cliente': matriz_preferencia.loc[CLIENTE_ID]
        , 'Participação geral': popularidade_categorias
    }).sort_values('Este cliente', ascending=False)
    comparacao_cliente.index.name = 'product_group_name'
    comparacao_cliente = comparacao_cliente.reset_index().melt(
        id_vars='product_group_name', var_name='Referencia', value_name='Participacao'
    )

    fig = px.bar(
        comparacao_cliente,
        x='Participacao',
        y='product_group_name',
        color='Referencia',
        orientation='h',
        barmode='group',
        color_discrete_map={'Este cliente': '#3274A1', 'Participação geral': '#C97932'},
        labels={'Participacao': 'Participação dos itens', 'product_group_name': 'Categoria'},
        title=f'Perfil individual e popularidade geral<br><sup>Cliente {CLIENTE_ID[:12]}… | {periodo_preferencia}</sup>'
    )
    fig.update_xaxes(tickformat='.0%', range=[0, 1])
    fig.update_yaxes(autorange='reversed')
    fig.update_layout(height=750, margin=dict(t=100))
    fig.show()
else:
    print('Escolha um CLIENTE_ID existente em perfil_preferencia.index.')

Cliente: 7f8c345bdeff3d1f17aaccf9b5fe01a2c20cda41e805b4eeaa1ef695cc3d9d0d
Itens: 11
Dias ativos: 2
Categorias na liderança: 1


,product_group_name,qtd_itens,dias_categoria,participacao_cliente,participacao_dias,participacao_geral,afinidade_relativa,posicao
200069,garment_lower_body,7,1,0.636,0.5,0.232,2.748,1.0
200070,garment_upper_body,3,1,0.273,0.5,0.5,0.546,2.0
200068,accessories,1,1,0.091,0.5,0.053,1.712,3.0


### Essa preferência reaparece depois?
Agora fazemos uma checagem temporal descritiva, **sem treinar modelo**. Reservamos os últimos sete dias desta EDA como período posterior. Toda categoria líder, popularidade e seleção por histórico é recalculada usando apenas os dias anteriores — não usamos os perfis completos para antecipar o futuro.

Comparamos a categoria líder de cada cliente com **uma mesma categoria mais popular do histórico**. Se houver empate na popularidade geral, usamos ordem alfabética. Clientes sem líder única ficam fora desta comparação; informamos quantos são. Os dois critérios serão avaliados nos mesmos clientes, com histórico mínimo e pelo menos um item no período posterior.

Quem não comprou depois não é tratado como rejeição. Novos clientes e históricos curtos são contados, mas não sustentam esta comparação de preferência individual. Assim, o resultado é condicionado aos clientes que voltaram a comprar e atende apenas esse grupo.

In [20]:
DIAS_POSTERIORES = 7
if DIAS_POSTERIORES < 1:
    raise ValueError('DIAS_POSTERIORES precisa ser pelo menos 1.')

data_corte_preferencia = df_preferencia['dia'].max() - pd.Timedelta(days=DIAS_POSTERIORES - 1)
historico_preferencia = df_preferencia.loc[df_preferencia['dia'] < data_corte_preferencia].copy()
posterior_preferencia = df_preferencia.loc[df_preferencia['dia'] >= data_corte_preferencia].copy()

if historico_preferencia.empty or posterior_preferencia.empty:
    raise ValueError('Amplie o recorte ou reduza DIAS_POSTERIORES para separar histórico e período posterior.')

print('Histórico:', historico_preferencia['dia'].min().date(), 'até', historico_preferencia['dia'].max().date())
print('Período posterior:', posterior_preferencia['dia'].min().date(), 'até', posterior_preferencia['dia'].max().date())

suporte_historico = historico_preferencia.groupby('customer_id').agg(
      itens_historico=('product_group_name', 'size')
    , dias_historico=('dia', 'nunique')
)
contagem_historica = historico_preferencia.groupby(['customer_id', 'product_group_name']).size().reset_index(name='qtd_itens')
contagem_historica['posicao'] = contagem_historica.groupby('customer_id')['qtd_itens'].rank(method='min', ascending=False)

lideres_historicos = contagem_historica.loc[contagem_historica['posicao'] == 1].copy()
lideres_historicos = lideres_historicos.merge(suporte_historico, on='customer_id', validate='many_to_one')
lideres_historicos = lideres_historicos.loc[
      (lideres_historicos['itens_historico'] >= MIN_ITENS_PREFERENCIA)
    & (lideres_historicos['dias_historico'] >= MIN_DIAS_PREFERENCIA)
].copy()

empatados_historico = lideres_historicos.loc[lideres_historicos.duplicated('customer_id', keep=False), 'customer_id'].nunique()
lider_historico_unico = lideres_historicos.drop_duplicates('customer_id', keep=False)

popularidade_historica = historico_preferencia['product_group_name'].value_counts().rename_axis('product_group_name').reset_index(name='qtd_itens')
popularidade_historica = popularidade_historica.sort_values(['qtd_itens', 'product_group_name'], ascending=[False, True])
categoria_popular_historico = popularidade_historica.iloc[0]['product_group_name']
print('Categoria mais popular no histórico:', categoria_popular_historico)
print('Clientes com histórico mínimo e liderança empatada:', empatados_historico)

Histórico: 2018-09-20 até 2018-10-03
Período posterior: 2018-10-04 até 2018-10-10


Categoria mais popular no histórico: garment_upper_body
Clientes com histórico mínimo e liderança empatada: 1937


### Repetição individual versus popularidade
Medimos duas coisas, dando o mesmo peso a cada cliente: **presença**, se a categoria apareceu em ao menos um item posterior; e **participação**, qual fração dos itens posteriores ficou naquela categoria, em média por cliente.

Também separamos clientes cuja líder histórica já era a categoria mais popular. Nesse grupo, os dois critérios são idênticos; o grupo com uma líder diferente é o mais informativo sobre personalização. Não calculamos significância nem tratamos uma única janela curta como validação definitiva.

In [21]:
itens_posteriores = posterior_preferencia.groupby('customer_id').size().rename('itens_posteriores')
contagem_posterior = posterior_preferencia.groupby(['customer_id', 'product_group_name']).size().reset_index(name='itens_categoria')

avaliacao_preferencias = lider_historico_unico[['customer_id', 'product_group_name']].rename(
    columns={'product_group_name': 'categoria_lider'}
).merge(itens_posteriores, on='customer_id', how='inner', validate='one_to_one')

avaliacao_preferencias = avaliacao_preferencias.merge(
      contagem_posterior.rename(columns={'product_group_name': 'categoria_lider', 'itens_categoria': 'itens_na_lider'})
    , on=['customer_id', 'categoria_lider']
    , how='left'
    , validate='one_to_one'
)

itens_na_popular = contagem_posterior.loc[
    contagem_posterior['product_group_name'] == categoria_popular_historico
].set_index('customer_id')['itens_categoria']

avaliacao_preferencias['itens_na_lider'] = avaliacao_preferencias['itens_na_lider'].fillna(0)
avaliacao_preferencias['itens_na_popular'] = avaliacao_preferencias['customer_id'].map(itens_na_popular).fillna(0)
avaliacao_preferencias['repetiu_lider'] = avaliacao_preferencias['itens_na_lider'] > 0
avaliacao_preferencias['comprou_popular'] = avaliacao_preferencias['itens_na_popular'] > 0
avaliacao_preferencias['participacao_lider_posterior'] = avaliacao_preferencias['itens_na_lider'] / avaliacao_preferencias['itens_posteriores']
avaliacao_preferencias['participacao_popular_posterior'] = avaliacao_preferencias['itens_na_popular'] / avaliacao_preferencias['itens_posteriores']
avaliacao_preferencias['grupo'] = np.where(
    avaliacao_preferencias['categoria_lider'] == categoria_popular_historico,
    'Líder igual à popular', 'Líder diferente da popular'
)

print('Clientes no período posterior:', len(itens_posteriores))
print('Destes, sem histórico anterior:', (~itens_posteriores.index.isin(suporte_historico.index)).sum())
print('Clientes elegíveis pelo histórico:', len(lider_historico_unico))
print('Elegíveis que voltaram a comprar:', len(avaliacao_preferencias))
print('Cobertura dos clientes posteriores: {:.1%}'.format(len(avaliacao_preferencias) / len(itens_posteriores)))

resumo_repeticao = pd.DataFrame({
      'Referencia': ['Líder individual', 'Mais popular no histórico']
    , 'Presença em algum item posterior': [avaliacao_preferencias['repetiu_lider'].mean(), avaliacao_preferencias['comprou_popular'].mean()]
    , 'Participação média nos itens posteriores': [avaliacao_preferencias['participacao_lider_posterior'].mean(), avaliacao_preferencias['participacao_popular_posterior'].mean()]
})

resumo_por_grupo = avaliacao_preferencias.groupby('grupo').agg(
      clientes=('customer_id', 'size')
    , presenca_lider=('repetiu_lider', 'mean')
    , presenca_popular=('comprou_popular', 'mean')
    , participacao_lider=('participacao_lider_posterior', 'mean')
    , participacao_popular=('participacao_popular_posterior', 'mean')
)

display(resumo_repeticao.round(3))
display(resumo_por_grupo.round(3))

if not avaliacao_preferencias.empty:
    fig = px.bar(
        resumo_repeticao.melt(id_vars='Referencia', var_name='Medida', value_name='Valor'),
        x='Medida', y='Valor', color='Referencia', barmode='group',
        text_auto='.1%',
        color_discrete_map={'Líder individual': '#3274A1', 'Mais popular no histórico': '#C97932'},
        title=f'Repetição de categoria no período posterior<br><sup>{posterior_preferencia["dia"].min():%d/%m/%Y} a {posterior_preferencia["dia"].max():%d/%m/%Y} | mesmos {len(avaliacao_preferencias):,} clientes nos dois critérios</sup>'
    )
    fig.update_yaxes(tickformat='.0%', range=[0, 1], title='Proporção')
    fig.update_layout(margin=dict(t=100), legend=dict(orientation='h', y=-0.3, x=0))
    fig.show()
else:
    print('Não há clientes elegíveis com compras posteriores para comparar os critérios.')

Clientes no período posterior: 72419
Destes, sem histórico anterior: 49019
Clientes elegíveis pelo histórico: 15947
Elegíveis que voltaram a comprar: 4110
Cobertura dos clientes posteriores: 5.7%


,Referencia,Presença em algum item posterior,Participação média nos itens posteriores
0,Líder individual,0.673,0.463
1,Mais popular no histórico,0.722,0.506


,clientes,presenca_lider,presenca_popular,participacao_lider,participacao_popular
grupo,,,,,
Líder diferente da popular,1328,0.474,0.627,0.280,0.413
Líder igual à popular,2782,0.767,0.767,0.551,0.551


### Leitura desta execução
Os números abaixo são recalculados junto com a EDA. A concentração no histórico e a repetição posterior respondem a perguntas diferentes; uma não garante a outra.

In [22]:
lideres_com_historico = lideres_unicas.loc[
    lideres_unicas['customer_id'].isin(clientes_com_historico.index)
]
print('Categoria mais comum no recorte:', popularidade_categorias.idxmax())
print('Participação dessa categoria nos itens: {:.1%}'.format(popularidade_categorias.max()))
print('Clientes com histórico mínimo e líder única:', len(lideres_com_historico))
print('Destes, com a líder presente em pelo menos dois dias: {:.1%}'.format(
    lideres_com_historico['dias_categoria'].ge(2).mean()
))

if not avaliacao_preferencias.empty:
    presenca_individual = avaliacao_preferencias['repetiu_lider'].mean()
    presenca_popular = avaliacao_preferencias['comprou_popular'].mean()
    participacao_individual = avaliacao_preferencias['participacao_lider_posterior'].mean()
    participacao_popular = avaliacao_preferencias['participacao_popular_posterior'].mean()
    print('Presença posterior: líder individual {:.1%}; popularidade {:.1%}.'.format(presenca_individual, presenca_popular))
    print('Participação média posterior: líder individual {:.1%}; popularidade {:.1%}.'.format(participacao_individual, participacao_popular))
    if presenca_individual > presenca_popular and participacao_individual > participacao_popular:
        print('Nesta janela e neste grupo, a líder individual superou a popularidade nas duas medidas.')
    elif presenca_individual <= presenca_popular and participacao_individual <= participacao_popular:
        print('Nesta janela e neste grupo, a líder individual não superou a popularidade nas duas medidas.')
    else:
        print('As duas medidas apontam direções diferentes; não há uma vantagem consistente nesta janela.')
    print('A comparação cobre {:.1%} dos clientes do período posterior; não é uma conclusão sobre toda a base.'.format(
        len(avaliacao_preferencias) / len(itens_posteriores)
    ))
else:
    print('Sem clientes comparáveis para interpretar repetição posterior.')

Categoria mais comum no recorte: garment_upper_body
Participação dessa categoria nos itens: 50.0%
Clientes com histórico mínimo e líder única: 28029
Destes, com a líder presente em pelo menos dois dias: 80.9%
Presença posterior: líder individual 67.3%; popularidade 72.2%.
Participação média posterior: líder individual 46.3%; popularidade 50.6%.
Nesta janela e neste grupo, a líder individual não superou a popularidade nas duas medidas.
A comparação cobre 5.7% dos clientes do período posterior; não é uma conclusão sobre toda a base.


### Como essa EDA orienta o próximo passo
A concentração descreve o histórico; a checagem posterior mostra se existe sinal de repetição **além da popularidade**. Olhe especialmente o grupo cuja categoria líder é diferente da mais popular.

Se a líder individual superar a referência nesse grupo, vale testar personalização por categoria. Se não superar, a preferência simples ainda não justificou sua vantagem nesta janela: podemos explorar recência, tipo de produto mais específico (`product_type_name`) e combinações de categorias. Isso não descarta modelos capazes de aprender relações mais complexas.

Antes de escolher um modelo, precisamos repetir a comparação em outras janelas e ampliar o histórico. Clientes sem histórico precisam de uma alternativa baseada em popularidade ou atributos disponíveis. E acertar um grupo amplo não equivale a acertar o `article_id` pedido numa recomendação de artigos.

Nenhuma etapa aqui treina um modelo. `preferencias_clientes`, `matriz_preferencia` e `avaliacao_preferencias` ficam disponíveis para continuar a investigação. Referências de código: [pandas — groupby](https://pandas.pydata.org/docs/user_guide/groupby.html) e [pivot](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html).